# Workshop: Deep Convolutional GAN (DCGAN)

1. **Core Concept**: Transition from MLPs to Convolutional hierarchies for image generation.
2. **Spatial Upsampling**: Understanding `ConvTranspose2d` as the engine of the Generator.
3. **Dataset Flexibility**: Training on color (RGB) images using custom folders.
4. **Stability Mechanisms**: The role of Batch Normalization and Strided Convolutions.

By the end of this lab, you will have implemented a Generator that can create 3-channel color images from random noise by competing against a Convolutional Discriminator.

## 1. Architectural Foundation

Standard GANs use simple dense layers. **DCGAN** (Deep Convolutional GAN) introduced several key constraints to make GAN training stable for high-resolution images:

1. **No Pooling Layers**: We replace MaxPool with **Strided Convolutions** in the Discriminator and **Fractionally-strided Convolutions** in the Generator.
2. **Batch Normalization**: Applied in both networks to stabilize gradients.
3. **Tanh Output**: The Generator uses a Tanh activation for the final image output (range -1 to 1).
4. **LeakyReLU**: Used in the Discriminator to prevent the "gradient death" problem common in adversarial training.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader, Dataset

import matplotlib.pyplot as plt
import numpy as np
import os

In [ ]:
# Configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Executing on: {device}')

# SETTINGS: Choose dataset mode here
USE_CIFAR10 = True  # Set to False to use 'my_image_paths' below

# Constants for typical DCGAN setup
image_size = 64  # DCGAN typically works best on powers of 2 (32, 64, 128)
nc = 3           # Number of color channels (RGB)
nz = 100         # Size of z latent vector (Generator input)
ngf = 64         # Size of feature maps in generator
ndf = 64         # Size of feature maps in discriminator

def weights_init(m):
    """
    Initialize weights for Conv and BatchNorm layers. 
    Typical DCGAN initialization.
    The weights are initialized to be close to 0 for stability.
    """
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

## 2. Flexible Image Intake

Instead of relying on rigid folder structures, we'll use a **Custom Dataset** class that takes a simple list of image paths.

In [ ]:
class CustomImageDataset(Dataset):
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform: image = self.transform(image)
        return image, 0

# 1. Define custom image paths
my_image_paths = [
    'd:/Universities/MSc-DLNN-Islington/Codes/Images/images_h.jpg',
    'd:/Universities/MSc-DLNN-Islington/Codes/img/images_a.jpg',
    'd:/Universities/MSc-DLNN-Islington/Codes/img/images_d.jpg'
]

# 2. Common Transforms
transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.CenterCrop(image_size),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

# 3. Load Dataset based on USE_CIFAR10 flag
if USE_CIFAR10:
    print('Loading CIFAR-10 Dataset...')
    dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
else:
    valid_paths = [p for p in my_image_paths if os.path.exists(p)]
    if not valid_paths:
        raise FileNotFoundError('No valid image paths found! Change USE_CIFAR10=True or update paths.')
    dataset = CustomImageDataset(valid_paths, transform=transform)
    print(f'Successfully loaded {len(dataset)} custom images.')

dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

In [ ]:
dataloader

## 3. The Generator ($G$)
The Generator is essentially a Convolutional network in reverse. It uses **Transposed Convolutions** to expand a single point of noise into a full $64 \times 64$ grid.

**Key mechanism**: `nn.ConvTranspose2d` increases the spatial resolution while reducing the number of feature maps.

In [ ]:

class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        self.main = nn.Sequential(
            # Input is noise Z (nz x 1 x 1)
            nn.ConvTranspose2d(nz, ngf * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),
            
            # State: (ngf*8) x 4 x 4
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            
            # State: (ngf*4) x 8 x 8
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            
            # State: (ngf*2) x 16 x 16
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            
            # State: (ngf) x 32 x 32 -> Final RGB Output: 3 x 64 x 64
            nn.ConvTranspose2d(ngf, nc, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, input):
        return self.main(input)

netG = Generator().to(device)
print(netG)

netG.apply(weights_init)


## 4. The Discriminator ($D$)
The Discriminator is a standard CNN classifier. Instead of classifying objects, it classifies probability ($0.0$ to $1.0$) of an image being "Real".

In [ ]:
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.main = nn.Sequential(
            # Input is (nc) x 64 x 64
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            
            # State: (ndf) x 32 x 32
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            
            # State: (ndf*2) x 16 x 16
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            
            # State: (ndf*4) x 8 x 8
            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
            
            # State: (ndf*8) x 4 x 4 -> Final probability score
            nn.Conv2d(ndf * 8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, input):
        return self.main(input).view(-1)

netD = Discriminator().to(device)
print(netD)

netD.apply(weights_init)

## WARNING
**Small Dataset Risk**: If we are training on only 3 - 30 images. 
In a commercial setting, DCGANs usually require >10,000 images. 
Training on a tiny set will cause the model to 'overfit' or 'collapse' to exactly those images. 
The augmentation added in Section 2 helps, but convergence will be fragile.

## 5. Optimal Hyperparameters
DCGANs are extremely sensitive to training settings. We use the settings recommended in the original Radford et al. paper.

In [ ]:

criterion = nn.BCELoss()
lr = 0.0002
beta1 = 0.5 # Momentum for Adam

optimizerD = optim.Adam(netD.parameters(), lr=lr, betas=(beta1, 0.999))
optimizerG = optim.Adam(netG.parameters(), lr=lr, betas=(beta1, 0.999))

fixed_noise = torch.FloatTensor(64, nz, 1, 1).normal_(0, 1) # batch_size = 64



In [ ]:

epochs = 2 # Try increasing and watch the images generated in every epoch cycle.

print("Starting Training Loop...")
for epoch in range(epochs):
    for i, data in enumerate(dataloader,0):
        
        # 1. Update D: Maximize log(D(x)) + log(1 - D(G(z)))
        netD.zero_grad()
        real_cpu = data[0].to(device)
        b_size = real_cpu.size(0)
        label = torch.full((b_size,), 1., device=device)
        output = netD(real_cpu)
        errD_real = criterion(output, label)
        errD_real.backward()

        noise = torch.randn(b_size, nz, 1, 1, device=device)
        fake = netG(noise)
        label.fill_(0.)
        output = netD(fake.detach())
        errD_fake = criterion(output, label)
        errD_fake.backward()
        optimizerD.step()

        # 2. Update G: Maximize log(D(G(z)))
        netG.zero_grad()
        label.fill_(1.)
        output = netD(fake)
        errG = criterion(output, label)
        errG.backward()
        optimizerG.step()

        if (i + 1) % 2 == 0:
            print(f'[{i+1}/{epochs}] Loss_D: {errD_real+errD_fake:.4f} Loss_G: {errG:.4f}')
    #     with torch.no_grad():
    #         samples = netG(fixed_noise).detach().cpu()
    #         grid = torchvision.utils.make_grid(samples, padding=2, normalize=True)
    #         plt.figure(figsize=(8,8))
    #         plt.imshow(np.transpose(grid,(1,2,0)))
    #         plt.axis('off')
    #         plt.show()


## 6. Result Visualization: Real vs. Generated
Let's create a utility to see how the model's 'hallucinations' compare to the real images you provided.

In [ ]:
def compare_real_and_fake(real_data, netG, noise):
    netG.eval()
    with torch.no_grad():
        fake = netG(noise).detach().cpu()
    
    # Get a batch of real images
    real_batch = next(iter(real_data))[0].cpu()
    
    plt.figure(figsize=(15, 8))
    
    # Plot Real
    plt.subplot(1, 2, 1)
    plt.axis('off')
    plt.title('Real Training Images')
    plt.imshow(np.transpose(torchvision.utils.make_grid(real_batch[:16], padding=5, normalize=True), (1, 2, 0)))
    
    # Plot Fake
    plt.subplot(1, 2, 2)
    plt.axis('off')
    plt.title('DCGAN Generated Images')
    plt.imshow(np.transpose(torchvision.utils.make_grid(fake[:16], padding=5, normalize=True), (1, 2, 0)))
    
    plt.show()

compare_real_and_fake(dataloader, netG, fixed_noise)

## 7. Summary
### Key Findings
- Compared to the standard MLP GAN, the DCGAN preserves **spatial structure** far better, leading to coherent textures and shapes.
- **Batch Normalization** is the 'secret sauce' that prevents the layers from having wild internal fluctuations during adversarial war.

### TASKS
1. **Resolution Scale**: Can you modify the Generator architecture to output $128 \times 128$ images?
2. **Learning Rate**: Observe what happens to the output if you increase the learning rate to `0.005` (Hint: Look for Mode Collapse).
3. **Color Channels**: Try converting the color images to grayscale and see if the training converges faster.
4. **Epochs**: Try training for more epochs and see if the training converges faster.
5. **Batch Size**: Try training with a batch size of `128` and see if the training converges faster.

Note: Carefully observe the output of the generator and discriminator.